<a href="https://colab.research.google.com/github/max7082/colab-practice/blob/main/mixup_%EA%B5%AC%ED%98%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import torch
from torch import nn
from torchvision.datasets import MNIST

device = "cuda" if torch.cuda.is_available() else "cpu"

data = MNIST("./data", train=True, download=True)
test_data = MNIST("./data", train=False, download=True)

# 학습 데이터 2000장과 원래 정답
x = data.data[:2000].float().reshape(2000, 784).to(device) / 255
label = data.targets[:2000].to(device)

# 정상 테스트 데이터 10000장
test_x = test_data.data.float().reshape(10000, 784).to(device) / 255
test_y = test_data.targets.to(device)

# 400장의 정답을 다른 숫자로 변경
# 네 방법 모두 같은 오류 정답을 사용하도록 한 번만 생성
torch.manual_seed(0)
noisy_label = label.clone()
wrong_idx = torch.randperm(len(label), device=device)[:400]
offset = torch.randint(1, 10, (len(wrong_idx),), device=device)
noisy_label[wrong_idx] = (label[wrong_idx] + offset) % 10

methods = ["ERM", "입력만 혼합", "정답만 혼합", "mixup"]
conditions = ["정상 정답", "정답 20% 오류"]
alpha = 1.0

for condition in conditions:
    print(f"\n========== {condition}로 학습 ==========")

    # 이번 실험에서 학습에 사용할 정답을 one-hot으로 변환
    if condition == "정상 정답":
        y = nn.functional.one_hot(label, num_classes=10).float()
    else:
        y = nn.functional.one_hot(noisy_label, num_classes=10).float()

    for method in methods:
        # 모든 조건과 방법에서 같은 초기 가중치로 시작
        torch.manual_seed(42)

        model = nn.Sequential(
            nn.Linear(784, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        ).to(device)

        optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

        # 학습
        for _ in range(500):
            lam = torch.distributions.Beta(alpha, alpha).sample().item()
            j = torch.randperm(len(x), device=device)

            mixed_x = lam * x + (1 - lam) * x[j]
            mixed_y = lam * y + (1 - lam) * y[j]

            if method == "ERM":
                input_x = x
                target_y = y

            elif method == "입력만 혼합":
                input_x = mixed_x
                target_y = y if lam >= 0.5 else y[j]

            elif method == "정답만 혼합":
                input_x = x
                target_y = mixed_y

            else:
                input_x = mixed_x
                target_y = mixed_y

            loss = nn.functional.cross_entropy(model(input_x), target_y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # 원래의 올바른 정답을 기준으로 정확도 계산
        model.eval()

        with torch.no_grad():
            train_prediction = model(x).argmax(1)
            train_accuracy = (
                (train_prediction == label).float().mean().item()
            )

            test_prediction = model(test_x).argmax(1)
            test_accuracy = (
                (test_prediction == test_y).float().mean().item()
            )

            # 정답을 바꾼 400장에서 틀린 학습 정답을 따른 비율
            if condition == "정답 20% 오류":
                wrong_label_accuracy = (
                    (train_prediction[wrong_idx] == noisy_label[wrong_idx])
                    .float().mean().item()
                )

        print(f"\n{method}")
        print(f"train accuracy (원래 정답 기준): {train_accuracy:.2%}")
        print(f"test accuracy: {test_accuracy:.2%}")

        if condition == "정답 20% 오류":
            print(
                f"변경한 400장에서 틀린 학습 정답과 일치한 비율: "
                f"{wrong_label_accuracy:.2%}"
            )

        # FGSM 공격: 각 모델의 기울기로 테스트 이미지 변경
        attack_x = test_x.detach().clone().requires_grad_(True)
        attack_loss = nn.functional.cross_entropy(model(attack_x), test_y)
        gradient = torch.autograd.grad(attack_loss, attack_x)[0]

        for epsilon in [0.05, 0.1, 0.2]:
            attacked_x = test_x + epsilon * gradient.sign()
            attacked_x = attacked_x.clamp(0, 1)

            with torch.no_grad():
                prediction = model(attacked_x).argmax(1)
                accuracy = (prediction == test_y).float().mean().item()

            print(
                f"epsilon={epsilon:.2f}: "
                f"FGSM 공격 후 정확도 {accuracy:.3%}"
            )


========== 정상 정답로 학습 ==========

ERM
train accuracy (원래 정답 기준): 95.00%
test accuracy: 89.30%
epsilon=0.05: FGSM 공격 후 정확도 69.230%
epsilon=0.10: FGSM 공격 후 정확도 31.230%
epsilon=0.20: FGSM 공격 후 정확도 1.300%

입력만 혼합
train accuracy (원래 정답 기준): 92.90%
test accuracy: 88.32%
epsilon=0.05: FGSM 공격 후 정확도 69.940%
epsilon=0.10: FGSM 공격 후 정확도 40.100%
epsilon=0.20: FGSM 공격 후 정확도 1.870%

정답만 혼합
train accuracy (원래 정답 기준): 91.50%
test accuracy: 87.70%
epsilon=0.05: FGSM 공격 후 정확도 68.460%
epsilon=0.10: FGSM 공격 후 정확도 38.650%
epsilon=0.20: FGSM 공격 후 정확도 1.930%

mixup
train accuracy (원래 정답 기준): 92.25%
test accuracy: 87.86%
epsilon=0.05: FGSM 공격 후 정확도 68.710%
epsilon=0.10: FGSM 공격 후 정확도 38.630%
epsilon=0.20: FGSM 공격 후 정확도 1.580%

========== 정답 20% 오류로 학습 ==========

ERM
train accuracy (원래 정답 기준): 91.85%
test accuracy: 86.15%
변경한 400장에서 틀린 학습 정답과 일치한 비율: 5.25%
epsilon=0.05: FGSM 공격 후 정확도 58.480%
epsilon=0.10: FGSM 공격 후 정확도 18.570%
epsilon=0.20: FGSM 공격 후 정확도 0.200%

입력만 혼합
train accuracy (원래 정답 기준): 90.35%
test 